In [ ]:
import torch
import pandas as pd
import numpy as np

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import pandas as pd
from pathlib import Path
import requests

# Create data directory
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Download CFPB complaint dataset
url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

zip_path = data_dir / "complaints.csv.zip"

if not zip_path.exists():
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)

print("Download completed!")



In [ ]:
import pandas as pd
import zipfile
from pathlib import Path

data_dir = Path("../data")

zip_path = next(
    data_dir.glob("CCDB_Export*.zip")
)

with zipfile.ZipFile(zip_path) as z:
    print("Files:", z.namelist())

    csv_name = next(
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    )

    with z.open(csv_name) as f:
        df = pd.read_csv(
            f,
            nrows=50000,
            low_memory=False
        )

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

In [ ]:
df = df[["Consumer complaint narrative", "Product"]].copy()

df = df.dropna()

df.columns = ["text", "label"]

print("Remaining samples:", len(df))
print(df["label"].value_counts())

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"_", " ", text)
    return text

df["text"] = df["text"].apply(clean_text)

df = df[df["text"].str.len() > 0].copy()

display(df.head())


In [ ]:
categories = [
    "Credit card",
    "Mortgage",
    "Debt collection",
    "Student loan"
]

df = df[df["label"].isin(categories)].copy()

print("Selected samples:", len(df))
print(df["label"].value_counts())

In [ ]:
df = (
    df.groupby("label", group_keys=False)
      .sample(n=500, random_state=42)
      .reset_index(drop=True)
)

print("Final dataset:", df.shape)
print(df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
from collections import Counter

MAX_VOCAB_SIZE = 10000

counter = Counter()

for text in train_df["text"]:
    counter.update(text.split())

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Example:", list(vocab.items())[:10])

In [ ]:
MAX_LEN = 100

def encode_text(text):
    tokens = text.split()[:MAX_LEN]

    ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    return ids

example = encode_text("my credit card was charged twice")

print("Encoded length:", len(example))
print("First 10 IDs:", example[:10])

In [ ]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

print("GloVe loaded!")
print("Vocabulary:", len(glove))
print("Embedding dimension:", glove.vector_size)

In [ ]:
import numpy as np

EMBEDDING_DIM = 50

embedding_matrix = np.random.normal(
    0, 0.1,
    (len(vocab), EMBEDDING_DIM)
).astype(np.float32)

embedding_matrix[vocab["<PAD>"]] = 0

found = 0

for word, idx in vocab.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]
        found += 1

print("Embedding matrix:", embedding_matrix.shape)
print("Words found:", found)
print("Coverage:", round(found / len(vocab) * 100, 2), "%")

In [ ]:
labels = sorted(df["label"].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print(label2id)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ComplaintDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].map(label2id).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text_ids = encode_text(self.texts[idx])

        return (
            torch.tensor(text_ids, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

In [ ]:
BATCH_SIZE = 32

train_dataset = ComplaintDataset(train_df)
val_dataset = ComplaintDataset(val_df)
test_dataset = ComplaintDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
texts, labels = next(iter(train_loader))

print("Text batch shape:", texts.shape)
print("Label batch shape:", labels.shape)

In [ ]:
import torch
import torch.nn as nn

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, hidden_dim, num_classes):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix),
            freeze=False,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)

        lengths = (x != 0).sum(dim=1).clamp(min=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed)

        output = self.fc(hidden[-1])

        return output

In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self, embedding_matrix, hidden_dim, num_classes):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix),
            freeze=False,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)

        lengths = (x != 0).sum(dim=1).clamp(min=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, cell) = self.lstm(packed)

        output = self.fc(hidden[-1])

        return output

In [ ]:
HIDDEN_DIM = 64
NUM_CLASSES = len(label2id)

rnn_model = RNNClassifier(
    embedding_matrix,
    HIDDEN_DIM,
    NUM_CLASSES
)

lstm_model = LSTMClassifier(
    embedding_matrix,
    HIDDEN_DIM,
    NUM_CLASSES
)

sample_texts, _ = next(iter(train_loader))

with torch.no_grad():
    rnn_output = rnn_model(sample_texts)
    lstm_output = lstm_model(sample_texts)

print("RNN output:", rnn_output.shape)
print("LSTM output:", lstm_output.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from tqdm.auto import tqdm

torch.manual_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EPOCHS = 10
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

print("Training device:", device)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for texts, labels in loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts)

        loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for texts, labels in loader:

            texts = texts.to(device)
            labels = labels.to(device)

            outputs = model(texts)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
def train_model(model, model_name):

    model = model.to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_acc = -1.0

    for epoch in tqdm(range(EPOCHS), desc=model_name):

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion
        )

        val_loss, val_acc = evaluate(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                model_dir / f"{model_name}.pt"
            )

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

    return history

In [ ]:
rnn_history = train_model(rnn_model, "rnn")

In [ ]:
lstm_history = train_model(lstm_model, "lstm")

In [ ]:
rnn_model.load_state_dict(
    torch.load(
        model_dir / "rnn.pt",
        map_location=device,
        weights_only=True
    )
)

lstm_model.load_state_dict(
    torch.load(
        model_dir / "lstm.pt",
        map_location=device,
        weights_only=True
    )
)

rnn_model.eval()
lstm_model.eval()

print("Best models loaded!")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(10, 6))

plt.plot(epochs, rnn_history["train_acc"],
         label="RNN Train", linestyle="--")

plt.plot(epochs, rnn_history["val_acc"],
         label="RNN Validation")

plt.plot(epochs, lstm_history["train_acc"],
         label="LSTM Train", linestyle="--")

plt.plot(epochs, lstm_history["val_acc"],
         label="LSTM Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("RNN vs LSTM Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    results_dir / "learning_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def get_predictions(model, loader):

    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for texts, labels in loader:

            texts = texts.to(device)

            outputs = model(texts)

            predictions = outputs.argmax(dim=1)

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_labels.extend(labels.tolist())

    return all_labels, all_predictions

In [ ]:
rnn_true, rnn_pred = get_predictions(
    rnn_model, test_loader
)

lstm_true, lstm_pred = get_predictions(
    lstm_model, test_loader
)

print("RNN predictions:", len(rnn_pred))
print("LSTM predictions:", len(lstm_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

def calculate_metrics(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, average="macro",
            zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, average="macro",
            zero_division=0
        ),
        "F1-score": f1_score(
            y_true, y_pred, average="macro",
            zero_division=0
        )
    }

results = pd.DataFrame({
    "RNN": calculate_metrics(rnn_true, rnn_pred),
    "LSTM": calculate_metrics(lstm_true, lstm_pred)
}).T

display(results.round(4))

results.to_csv(
    results_dir / "model_comparison.csv"
)

In [ ]:
class_names = [
    id2label[i] for i in range(NUM_CLASSES)
]

print("RNN Classification Report:\n")

print(classification_report(
    rnn_true,
    rnn_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("LSTM Classification Report:\n")

print(classification_report(
    lstm_true,
    lstm_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

for name, y_true, y_pred in [
    ("RNN", rnn_true, rnn_pred),
    ("LSTM", lstm_true, lstm_pred)
]:

    cm = confusion_matrix(
        y_true, y_pred,
        labels=list(range(NUM_CLASSES))
    )

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"{name} Confusion Matrix")
    plt.tight_layout()

    plt.savefig(
        results_dir / f"{name.lower()}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
def predict_complaint(text, model):

    model.eval()

    cleaned = clean_text(text)

    encoded = encode_text(cleaned)

    tensor = torch.tensor(
        [encoded],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        output = model(tensor)

        probabilities = torch.softmax(
            output, dim=1
        )

        predicted_id = probabilities.argmax(
            dim=1
        ).item()

        confidence = probabilities.max().item()

    return {
        "text": text,
        "prediction": id2label[predicted_id],
        "confidence": round(confidence, 4)
    }

In [ ]:
new_complaints = [
    "My credit card was charged twice for the same purchase.",

    "A debt collector keeps calling me about a debt I do not owe.",

    "My mortgage company increased my monthly home loan payment.",

    "I cannot afford to repay my student loans after graduation."
]

for text in new_complaints:

    result = predict_complaint(
        text, lstm_model
    )

    print("Complaint:", result["text"])
    print("Prediction:", result["prediction"])
    print("Confidence:", result["confidence"])
    print("-" * 60)

In [ ]:
prediction_results = []

for text in new_complaints:

    rnn_result = predict_complaint(text, rnn_model)

    lstm_result = predict_complaint(text, lstm_model)

    prediction_results.append({
        "Complaint": text,
        "RNN Prediction": rnn_result["prediction"],
        "RNN Confidence": rnn_result["confidence"],
        "LSTM Prediction": lstm_result["prediction"],
        "LSTM Confidence": lstm_result["confidence"]
    })

prediction_df = pd.DataFrame(prediction_results)

display(prediction_df)

prediction_df.to_csv(
    results_dir / "sample_predictions.csv",
    index=False
)

In [ ]:
import json

model_config = {
    "vocab": vocab,
    "label2id": label2id,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_classes": NUM_CLASSES
}

with open(
    model_dir / "model_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(model_config, f, indent=4)

print("Model configuration saved!")